# Interpolation des courbes de lumière (mTAN, LSST g r i z)

Ce notebook montre, pour un split (`val` / `test`), les courbes de lumière **observées** et la **reconstruction du modèle**,
dé-normalisées en magnitudes et en jours. Il utilise uniquement les sorties de `evaluate_unsupervised_lsst.py` :

- `evaluate_<split>_decoded_lcs_<TAG>.npz` et `evaluate_<split>_objIds_dataloader_<TAG>.npy`
- `min_max_mags_<TAG>.npy` et `train_min_max_times_<TAG>.npy` (produits par le preprocessing)
- *(optionnel)* `alerts_processed_<TAG>.parquet` : barres d'erreur σ et **résidus normalisés** `(obs − pred)/σ`

**Comment lire les figures.**
Le décodeur est évalué aux **temps d'observation** (de toutes les bandes) : la courbe de chaque bande est donc la prédiction du modèle en ces temps, reliée par une ligne.
Elle est tracée en **trait plein** entre la première et la dernière observation de la bande, et en **pointillés** en dehors (extrapolation).
Points = observations ; barres = σ des alertes (si disponibles).

## 0. Paramètres

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# ------------------------- À ADAPTER -------------------------
TAG = "ftransfer_lsst_2026_06_29_882968"
SETTING = "test"          # 'test', 'val' ou 'train' (le .npz correspondant doit exister)
N_SHOW = 9                # courbes par figure
NCOLS = 3
SEED = 0
GROUPS = ["aléatoires", "meilleures", "médianes", "pires", "plus denses", "plus courtes"]
USE_ALERTS = True         # barres d'erreur + résidus normalisés (nécessite le parquet)
ALERTS_PARQUET = f"alerts_processed_{TAG}.parquet"
FIG_DIR = "figures_interp"
# --------------------------------------------------------------

BANDS = ["g", "r", "i", "z"]
COLORS = ["#1b9e77", "#d95f02", "#7570b3", "#e7298a"]
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "figure.constrained_layout.use": True})
rng = np.random.default_rng(SEED)

## 1. Lecture et dé-normalisation

In [ ]:
dec = np.load(f"evaluate_{SETTING}_decoded_lcs_{TAG}.npz")
keys = sorted(dec.files, key=lambda k: int(k.split("_")[1]))
ids = np.load(f"evaluate_{SETTING}_objIds_dataloader_{TAG}.npy", allow_pickle=True)
assert len(keys) == len(ids), (len(keys), len(ids))

mm = np.load(f"min_max_mags_{TAG}.npy", allow_pickle=True)          # lignes : [objId, min_mag, max_mag]
min_mag = {str(r[0]): float(r[1]) for r in mm}
tmin, tmax = [float(x) for x in np.load(f"train_min_max_times_{TAG}.npy")[:2]]   # heures
print(f"{len(ids)} courbes ({SETTING}) ; temps d'entraînement : [{tmin:.1f}, {tmax:.1f}] h = {tmax/24:.0f} jours")

# --- σ des alertes (optionnel) : aligné ligne à ligne avec la courbe (tri par jd, comme get_lc)
groups = {}
if USE_ALERTS and os.path.exists(ALERTS_PARQUET):
    try:
        al = pd.read_parquet(ALERTS_PARQUET, columns=["objectId", "fid", "jd", "sigmapsf"])
        al["oid"] = al["objectId"].astype(str)
        al = al[al.oid.isin(set(map(str, ids)))].sort_values(["oid", "jd"], kind="stable")
        groups = {k: g for k, g in al.groupby("oid")}
        sg = al["sigmapsf"].to_numpy(float)
        print(f"σ lus pour {len(groups)} objets ; σ (mag) : médiane {np.nanmedian(sg):.3f}, "
              f"p5-p95 = [{np.nanpercentile(sg, 5):.3f}, {np.nanpercentile(sg, 95):.3f}], max {np.nanmax(sg):.3f}")
        if np.nanmax(sg) > 0.3:
            print("ATTENTION : avec snr_min=5 on attend σ_mag <= ~0.22. Vérifie que sigmapsf = 1.0857*psfFluxErr/psfFlux.")
    except Exception as e:
        print("Parquet illisible, barres d'erreur désactivées :", repr(e))
        groups = {}
elif USE_ALERTS:
    print(f"{ALERTS_PARQUET} introuvable : pas de barres d'erreur ni de résidus normalisés.")

def extract(i):
    a = dec[keys[i]]                                   # (4, T, dim) : pred, obs, mask, temps (répété)
    pred, obs, mask, tt = a[0], a[1], a[2], a[3][:, 0]
    valid = mask.sum(1) > 0                            # lignes réelles (le reste est du padding)
    pred, obs, mask, tt = pred[valid], obs[valid], mask[valid], tt[valid]
    oid = str(ids[i]); m0 = min_mag[oid]
    lc = dict(
        id=oid, dim=pred.shape[1],
        t=(tt * (tmax - tmin) + tmin) / 24.0,          # jours depuis la 1re alerte
        obs=np.where(mask > 0, obs * 2.5 + m0, np.nan),   # mag = x_norm * 2.5 + mag_min(courbe)
        pred=pred * 2.5 + m0, mask=mask, sigma=None)
    g = groups.get(oid)
    if g is not None and len(g) == len(lc["t"]) and np.array_equal(g["fid"].to_numpy(int) - 1, mask.argmax(1)):
        lc["sigma"] = g["sigmapsf"].to_numpy(float)
    return lc

lcs = [extract(i) for i in range(len(ids))]
if groups:
    n_ok = sum(l["sigma"] is not None for l in lcs)
    print(f"σ alignés avec les courbes : {n_ok}/{len(lcs)}")

## 2. Métriques par courbe

In [ ]:
rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for l in lcs:
        err = l["obs"] - l["pred"]
        r = dict(id=l["id"], n=int(l["mask"].sum()), duree_j=float(l["t"].max() - l["t"].min()),
                 rmse=float(np.sqrt(np.nanmean(err ** 2))))
        for b in range(l["dim"]):
            r[f"n_{BANDS[b]}"] = int(l["mask"][:, b].sum())
            r[f"rmse_{BANDS[b]}"] = float(np.sqrt(np.nanmean(err[:, b] ** 2))) if l["mask"][:, b].sum() else np.nan
        rows.append(r)
tab = pd.DataFrame(rows)
print(f"RMSE par courbe (mag) : médiane {tab.rmse.median():.3f}, moyenne {tab.rmse.mean():.3f}, p90 {tab.rmse.quantile(.9):.3f}, max {tab.rmse.max():.3f}")
tab.describe().loc[["mean", "50%", "max"]].round(3)

## 3. Figures

In [ ]:
def plot_lc_interp(ax, lc, errbars=True):
    t = lc["t"]
    for b in range(lc["dim"]):
        o = lc["obs"][:, b]; ok = ~np.isnan(o)
        if ok.sum() == 0:
            continue
        c = COLORS[b]
        p = lc["pred"][:, b]
        inside = (t >= t[ok].min()) & (t <= t[ok].max())
        ax.plot(t, p, ":", color=c, lw=1.0, alpha=0.7)                       # extrapolation
        ax.plot(t[inside], p[inside], "-", color=c, lw=1.7)                  # interpolation
        if errbars and lc["sigma"] is not None:
            ax.errorbar(t[ok], o[ok], yerr=lc["sigma"][ok], fmt="o", ms=3.5, color=c, alpha=0.85, elinewidth=0.8, capsize=0)
        else:
            ax.plot(t[ok], o[ok], "o", ms=3.5, color=c, alpha=0.85)
    ax.invert_yaxis()

def select(group, k):
    order = np.argsort(tab.rmse.values)
    if group == "aléatoires":   return rng.choice(len(tab), size=min(k, len(tab)), replace=False)
    if group == "meilleures":   return order[:k]
    if group == "pires":        return order[::-1][:k]
    if group == "médianes":
        m = len(order) // 2; return order[max(0, m - k // 2): max(0, m - k // 2) + k]
    if group == "plus denses":  return np.argsort(-tab.n.values)[:k]
    if group == "plus courtes": return np.argsort(tab.duree_j.values)[:k]
    raise ValueError(group)

def grid(group):
    idx = select(group, N_SHOW)
    nrows = int(np.ceil(len(idx) / NCOLS))
    fig, axs = plt.subplots(nrows, NCOLS, figsize=(4.4 * NCOLS, 3.0 * nrows), squeeze=False)
    for ax in axs.ravel():
        ax.set_visible(False)
    for ax, i in zip(axs.ravel(), idx):
        ax.set_visible(True)
        plot_lc_interp(ax, lcs[i])
        r = tab.iloc[i]
        ax.set_title(f"{r.id} | N={r.n} | RMSE={r.rmse:.2f}", fontsize=8)
        ax.tick_params(labelsize=8)
    for ax in axs[-1]:
        ax.set_xlabel("temps (jours)", fontsize=9)
    for ax in axs[:, 0]:
        ax.set_ylabel("mag", fontsize=9)
    handles = [plt.Line2D([], [], color=COLORS[b], marker="o", ls="-", ms=4, label=BANDS[b]) for b in range(4)]
    handles += [plt.Line2D([], [], color="gray", ls="-", label="interpolation"), plt.Line2D([], [], color="gray", ls=":", label="extrapolation")]
    fig.legend(handles=handles, loc="outside upper center", ncol=6, fontsize=9, frameon=False,
               title=f"Courbes {group} ({SETTING})", title_fontsize=11)
    path = os.path.join(FIG_DIR, f"interp_{SETTING}_{group.replace(' ', '_')}.png")
    fig.savefig(path, bbox_inches="tight"); print("figure enregistrée :", path)
    plt.show()

for g in GROUPS:
    grid(g)

## 4. Résidus normalisés par σ

`z = (m_obs − m_pred) / σ_psf`, sur tous les points observés. Repères :

- **écart-type de z ≈ 1** et χ² réduit ≈ 1 : le modèle reconstruit au niveau du bruit photométrique ;
- **< 1** : il suit le bruit de près (sur-ajustement, ou σ surestimés) ;
- **> 1** : il est plus lisse que les données (ou σ sous-estimés) : sous-ajustement possible sur des variations réelles.

Les incertitudes ne sont pas modélisées par mTAN (le modèle ne voit jamais σ) : ce diagnostic est donc indépendant de l'entraînement.

In [ ]:
zs, mags_z, band_z = [], [], []
for l in lcs:
    if l["sigma"] is None:
        continue
    ok = ~np.isnan(l["obs"])
    z = (l["obs"] - l["pred"]) / l["sigma"][:, None]
    for b in range(l["dim"]):
        m = ok[:, b] & np.isfinite(z[:, b])
        zs.append(z[m, b]); mags_z.append(l["obs"][m, b]); band_z.append(np.full(m.sum(), b))

if not zs:
    print("Pas de σ disponibles : section ignorée.")
else:
    z = np.concatenate(zs); mg = np.concatenate(mags_z); bz = np.concatenate(band_z)
    mad_std = lambda x: 1.4826 * np.median(np.abs(x - np.median(x)))
    summary = pd.DataFrame([dict(bande=BANDS[b], n=int((bz == b).sum()), moyenne=z[bz == b].mean(), std=z[bz == b].std(),
                                 std_robuste=mad_std(z[bz == b]), chi2_red=(z[bz == b] ** 2).mean(),
                                 frac_abs_lt1=(np.abs(z[bz == b]) < 1).mean(), frac_abs_lt2=(np.abs(z[bz == b]) < 2).mean())
                            for b in range(4) if (bz == b).any()]
                           + [dict(bande="toutes", n=len(z), moyenne=z.mean(), std=z.std(), std_robuste=mad_std(z),
                                   chi2_red=(z ** 2).mean(), frac_abs_lt1=(np.abs(z) < 1).mean(), frac_abs_lt2=(np.abs(z) < 2).mean())])
    display(summary.round(3))
    print("Repères gaussiens : |z|<1 -> 0.683 ; |z|<2 -> 0.954")

    fig, axs = plt.subplots(1, 3, figsize=(16, 4.4))
    ax = axs[0]
    xx = np.linspace(-6, 6, 400)
    ax.hist(np.clip(z, -6, 6), bins=80, density=True, alpha=0.6, color="C0", label="z observé (tronqué à ±6)")
    ax.plot(xx, np.exp(-xx ** 2 / 2) / np.sqrt(2 * np.pi), "k", lw=1.5, label="N(0, 1)")
    ax.set_xlabel("(obs − pred) / σ"); ax.set_ylabel("densité"); ax.set_title("Résidus normalisés"); ax.legend(loc="upper right")

    ax = axs[1]
    for b in range(4):
        m = bz == b
        if m.any():
            ax.hist(np.clip(z[m], -6, 6), bins=60, histtype="step", density=True, color=COLORS[b], lw=1.8, label=BANDS[b])
    ax.set_xlabel("(obs − pred) / σ"); ax.set_title("Par bande"); ax.legend()

    ax = axs[2]
    hb = ax.hexbin(mg, np.clip(z, -6, 6), gridsize=40, mincnt=1, cmap="viridis", bins="log")
    ax.axhline(0, color="w", lw=0.8); fig.colorbar(hb, ax=ax, label="comptes (log)")
    ax.set_xlabel("magnitude observée"); ax.set_ylabel("z"); ax.set_title("z en fonction de la magnitude")
    path = os.path.join(FIG_DIR, f"residus_normalises_{SETTING}.png"); fig.savefig(path, bbox_inches="tight"); print("figure enregistrée :", path)
    plt.show()